<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/05_WOA_Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۵: اجرای نهایی انتخاب ویژگی با الگوریتم نهنگ

در نوت‌بوک ۰۴ الگوریتم را ساختیم و یک بار با یک seed آزمودیم. این‌جا همان الگوریتم را **چند بار با seedهای مختلف** اجرا می‌کنیم و بهترین نتیجه را به‌عنوان انتخاب نهایی ذخیره می‌کنیم.

**چرا چند seed؟** الگوریتم نهنگ تصادفی است (جمعیت اولیه و حرکت‌ها تصادفی‌اند). یک اجرا ممکن است شانسی خوب یا بد باشد. با چند اجرای مستقل می‌بینیم نتیجه چقدر پایدار است و بهترین را برمی‌داریم.

مراحل: آماده‌سازی داده و مدل ارزیاب، تعریف دوباره‌ی توابع WOA (همان توابع ۰۴، بدون تغییر)، تنظیم پارامترها، اجرای هر seed، مقایسه‌ی اجراها، انتخاب بهترین و ذخیره‌ی نتایج.

**نکته‌ی روش‌شناسی:** بهترین seed فقط با **fitness روی train (CV)** انتخاب می‌شود و test در هیچ بخشی دخالت ندارد. ارزیابی روی test در نوت‌بوک ۰۶ انجام می‌شود.

</div>

<div dir="rtl" style="text-align: right">

### اتصال به Google Drive
Drive وصل می‌شود تا به دیتاست‌ها، `dataset_config.py` و پوشه‌ی نتایج دسترسی داشته باشیم.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<div dir="rtl" style="text-align: right">

### وارد کردن کتابخانه‌ها
`numpy` و `pandas` برای محاسبات و جدول‌ها، `matplotlib` برای نمودار، و از `scikit-learn` ابزارهای مدل ارزیاب (`Pipeline`، `StandardScaler`، `LogisticRegression`) و `cross_val_score` برای fitness.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Libraries imported successfully.")

<div dir="rtl" style="text-align: right">

### بارگذاری و آماده‌سازی داده
مثل نوت‌بوک ۰۴:
- تنظیمات دیتاست وارد و پوشه‌ی نتایج (`results/<نام دیتاست>/`) ساخته می‌شود.
- داده خوانده، به ویژگی و برچسب جدا و به train و test تقسیم می‌شود (برای Parkinson بر اساس فرد).
- ویژگی‌های تکراری و ثابت فقط از روی train حذف می‌شوند.
- `n_features` تعداد ویژگی‌های باقی‌مانده است و طول هر نهنگ را تعیین می‌کند.

ابعاد باید با نوت‌بوک‌های ۰۳ و ۰۴ یکی باشد.

</div>

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")

import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

RESULTS_PATH = get_results_path(PROJECT_PATH)

df = load_dataset(PROJECT_PATH)

X, y, groups = split_xy(df)

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

n_features = X_train.shape[1]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Number of features:", n_features)

<div dir="rtl" style="text-align: right">

### مدل ارزیاب، CV و ضریب $\alpha$
- `classifier`: Pipeline شامل `StandardScaler` و Logistic Regression (با `class_weight="balanced"`). این مدل در هر ارزیابی fitness استفاده می‌شود.
- `cv`: اعتبارسنجی ۵-تایی روی train (برای Parkinson بر اساس فرد).
- `alpha = 0.9`: وزن خطای مدل در fitness؛ وزن تعداد ویژگی‌ها $1-\alpha = 0.1$ است.

همان تنظیمات نوت‌بوک ۰۴ است تا نتایج قابل مقایسه باشند.

</div>

In [ ]:
classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

cv = make_cv(groups_train)

alpha = 0.9

print("Classifier and CV are ready.")

<div dir="rtl" style="text-align: right">

### تعریف توابع الگوریتم نهنگ
همه‌ی توابع نوت‌بوک ۰۴ یک‌جا در این سلول آمده‌اند تا این نوت‌بوک مستقل اجرا شود:
- `sigmoid` و `continuous_to_binary` / `population_to_binary`: تبدیل موقعیت پیوسته به بردار صفر و یک با تابع انتقال S-شکل.
- `calculate_coefficients`: محاسبه‌ی $a$، $A$ و $C$.
- `update_whale_positions`: به‌روزرسانی موقعیت نهنگ‌ها با سه رفتار محاصره، جست‌وجو و مارپیچ.
- `fitness_function`: $\alpha\,(1-F1_{macro}) + (1-\alpha)\,k/n$ با CV روی train؛ مقدار کمتر بهتر است.
- `binary_woa`: حلقه‌ی اصلی الگوریتم که بهترین بردار باینری، بهترین fitness و منحنی همگرایی را برمی‌گرداند.

توضیح کامل هر تابع در نوت‌بوک ۰۴ آمده است. این سلول فقط تعریف می‌کند و چیزی اجرا نمی‌کند.

</div>

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))


def continuous_to_binary(position, rng):
    probabilities = sigmoid(position)

    random_values = rng.random(size=position.shape)

    binary_position = (random_values < probabilities).astype(int)

    return binary_position

def calculate_coefficients(iteration, max_iterations, rng):
    a = 2 - iteration * (2 / max_iterations)

    r1 = rng.random()
    r2 = rng.random()

    A = 2 * a * r1 - a
    C = 2 * r2

    return A, C

def update_whale_positions(
    positions,
    best_position,
    iteration,
    max_iterations,
    rng,
    lower_bound=-6,
    upper_bound=6
):
    new_positions = np.zeros_like(positions)

    a = 2 - iteration * (2 / max_iterations)

    n_whales = positions.shape[0]

    for i in range(n_whales):

        X = positions[i].copy()

        r1 = rng.random()
        r2 = rng.random()

        A = 2 * a * r1 - a
        C = 2 * r2

        p = rng.random()

        if p < 0.5:

            if abs(A) < 1:

                # Encircling the best solution
                D = np.abs(C * best_position - X)
                new_position = best_position - A * D

            else:

                # Search for a random whale
                random_index = rng.integers(0, n_whales)
                random_position = positions[random_index]

                D = np.abs(C * random_position - X)
                new_position = random_position - A * D

        else:

            # Spiral movement
            b = 1
            l = rng.uniform(-1, 1)

            D_prime = np.abs(best_position - X)

            new_position = (
                D_prime * np.exp(b * l) * np.cos(2 * np.pi * l)
                + best_position
            )

        # Keep positions within the search boundaries
        new_positions[i] = np.clip(
            new_position,
            lower_bound,
            upper_bound
        )

    return new_positions

def population_to_binary(positions, rng):
    probabilities = sigmoid(positions)

    random_values = rng.random(size=positions.shape)

    binary_population = (
        random_values < probabilities
    ).astype(int)

    return binary_population

def fitness_function(
    binary_position,
    X_train,
    y_train,
    classifier,
    cv,
    alpha=0.9
):
    # Convert input to a NumPy array
    binary_position = np.asarray(binary_position)

    # Find selected feature indices
    selected_indices = np.where(binary_position == 1)[0]

    # Count selected features
    n_selected = len(selected_indices)

    # Reject solutions with no selected features
    if n_selected == 0:
        return 1.0

    # Select only the chosen features
    X_selected = X_train.iloc[:, selected_indices]

    # Evaluate classification performance using cross-validation
    cv_scores = cross_val_score(
        classifier,
        X_selected,
        y_train,
        cv=cv,
        scoring="f1_macro"
    )

    mean_f1 = cv_scores.mean()

    # Calculate feature selection ratio
    feature_ratio = n_selected / X_train.shape[1]

    # Calculate final fitness
    fitness = (
        alpha * (1 - mean_f1)
        + (1 - alpha) * feature_ratio
    )

    return fitness

def binary_woa(
    X_train,
    y_train,
    classifier,
    cv,
    population_size=10,
    max_iterations=20,
    alpha=0.9,
    lower_bound=-6,
    upper_bound=6,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    n_features = X_train.shape[1]

    # 1. Initialize continuous whale positions
    positions = rng.uniform(
        lower_bound,
        upper_bound,
        size=(population_size, n_features)
    )

    # 2. Convert initial population to binary
    binary_population = population_to_binary(
        positions,
        rng
    )

    # 3. Evaluate initial population
    fitness_values = np.array([
        fitness_function(
            binary_population[i],
            X_train,
            y_train,
            classifier,
            cv,
            alpha
        )
        for i in range(population_size)
    ])

    # 4. Find initial global best
    best_index = np.argmin(fitness_values)

    best_fitness = fitness_values[best_index]
    best_position = positions[best_index].copy()
    best_binary_solution = binary_population[best_index].copy()

    convergence_curve = [best_fitness]

    # 5. Main WOA iterations
    for iteration in range(max_iterations):

        # Update continuous positions
        positions = update_whale_positions(
            positions=positions,
            best_position=best_position,
            iteration=iteration,
            max_iterations=max_iterations,
            rng=rng,
            lower_bound=lower_bound,
            upper_bound=upper_bound
        )

        # Convert updated positions to binary
        binary_population = population_to_binary(
            positions,
            rng
        )

        # Evaluate new population
        fitness_values = np.array([
            fitness_function(
                binary_population[i],
                X_train,
                y_train,
                classifier,
                cv,
                alpha
            )
            for i in range(population_size)
        ])

        # Find best candidate in current iteration
        current_best_index = np.argmin(fitness_values)
        current_best_fitness = fitness_values[current_best_index]

        # Update global best if improvement is found
        if current_best_fitness < best_fitness:

            best_fitness = current_best_fitness

            best_position = positions[current_best_index].copy()

            best_binary_solution = (
                binary_population[current_best_index].copy()
            )

        # Store best fitness so far
        convergence_curve.append(best_fitness)

        print(
            f"Iteration {iteration + 1}/{max_iterations} "
            f"| Best Fitness: {best_fitness:.6f} "
            f"| Selected Features: {best_binary_solution.sum()}"
        )

    return (
        best_binary_solution,
        best_fitness,
        convergence_curve
    )

print("WOA functions are ready.")

<div dir="rtl" style="text-align: right">

### تنظیم پارامترهای اجرا
جمعیت، تعداد تکرارها و فهرست seedها از `WOA_SETTINGS` در `dataset_config.py` برای دیتاست فعال خوانده می‌شود (پیش‌فرض: ۱۰ نهنگ، ۲۰ تکرار، seedهای ۴۲ و ۱۰۰ و ۲۰۲۶). برای تغییر آن‌ها فقط همان فایل را عوض کنید. هر seed یک اجرای مستقل با جمعیت اولیه‌ی متفاوت است.

</div>

In [ ]:
population_size = WOA_SETTINGS[ACTIVE_DATASET]["population_size"]
max_iterations = WOA_SETTINGS[ACTIVE_DATASET]["max_iterations"]

random_seeds = WOA_SETTINGS[ACTIVE_DATASET]["random_seeds"]

print("Population size:", population_size)
print("Maximum iterations:", max_iterations)
print("Random seeds:", random_seeds)

<div dir="rtl" style="text-align: right">

### اجرای الگوریتم برای هر seed
برای هر seed، `binary_woa` یک بار اجرا می‌شود (با همه‌ی پارامترها و مرزهای ‎−۶ و ‎+۶). نتیجه‌ی هر اجرا در فهرست `all_runs` ذخیره می‌شود: seed، بهترین بردار باینری، بهترین fitness، تعداد ویژگی‌های انتخاب‌شده و منحنی همگرایی. `.copy()` تضمین می‌کند اجراهای بعدی نتایج قبلی را تغییر ندهند. این بخش طولانی‌ترین قسمت نوت‌بوک است و پیشرفت هر تکرار چاپ می‌شود.

</div>

In [ ]:
all_runs = []

for seed in random_seeds:

    print(f"\nStarting WOA run with seed {seed}")

    best_solution, best_fitness, convergence_curve = binary_woa(
        X_train=X_train,
        y_train=y_train,
        classifier=classifier,
        cv=cv,
        population_size=population_size,
        max_iterations=max_iterations,
        alpha=alpha,
        lower_bound=-6,
        upper_bound=6,
        random_state=seed
    )

    all_runs.append({
        "seed": seed,
        "best_solution": best_solution.copy(),
        "best_fitness": best_fitness,
        "selected_features": int(best_solution.sum()),
        "convergence_curve": convergence_curve.copy()
    })

print("\nAll WOA runs completed.")

<div dir="rtl" style="text-align: right">

### جدول مقایسه‌ی اجراها
بهترین fitness و تعداد ویژگی‌های انتخاب‌شده‌ی هر seed در یک جدول نمایش داده می‌شود. اگر fitnessها نزدیک هم باشند ولی تعداد ویژگی‌ها خیلی فرق کند، یعنی چند زیرمجموعه‌ی متفاوت کیفیت تقریباً برابر دارند.

</div>

In [ ]:
runs_summary = pd.DataFrame([
    {
        "Seed": run["seed"],
        "Best Fitness": run["best_fitness"],
        "Selected Features": run["selected_features"]
    }
    for run in all_runs
])

runs_summary

<div dir="rtl" style="text-align: right">

### انتخاب بهترین اجرا
اجرایی که کمترین fitness را دارد (`argmin`) به‌عنوان نتیجه‌ی نهایی برگزیده می‌شود و بردار باینری و fitnessش در `final_woa_solution` و `final_woa_fitness` نگه داشته می‌شود. این انتخاب فقط با fitness روی train انجام می‌شود، نه test.

</div>

In [ ]:
best_run_index = np.argmin([
    run["best_fitness"] for run in all_runs
])

best_run = all_runs[best_run_index]

final_woa_solution = best_run["best_solution"].copy()
final_woa_fitness = best_run["best_fitness"]

print("Best seed:", best_run["seed"])
print("Best Fitness:", final_woa_fitness)
print("Selected features:", final_woa_solution.sum())

<div dir="rtl" style="text-align: right">

### ذخیره‌ی نتایج
- نام ویژگی‌های انتخاب‌شده‌ی بهترین اجرا (ستون‌هایی با مقدار ۱) در `final_woa_selected_features.csv`.
- جدول مقایسه‌ی اجراها در `woa_multiple_runs_summary.csv`.

هر دو در `results/<نام دیتاست>/` ذخیره می‌شوند. نوت‌بوک ۰۶ فایل اول را می‌خواند و ویژگی‌های انتخاب‌شده را روی test ارزیابی می‌کند.

</div>

In [ ]:
final_selected_features = X_train.columns[
    final_woa_solution == 1
].tolist()

pd.DataFrame({
    "Feature": final_selected_features
}).to_csv(
    RESULTS_PATH / "final_woa_selected_features.csv",
    index=False
)

runs_summary.to_csv(
    RESULTS_PATH / "woa_multiple_runs_summary.csv",
    index=False
)

print("Best selected features saved.")
print("Multiple runs summary saved.")